# 07 · Evaluación comparativa: nuestro modelo frente al Benchmarking

| | |
|---|---|
| **Fase CRISP-DM** | 5 — Evaluación |
| **Objetivo** | Comparar, en las mismas condiciones, los 4 modelos mejor puntuados del Benchmarking, nuestras variantes y 2 baselines, y comprobar si el ranking empírico coincide con el de la literatura. |
| **Entradas** | Gold: `series_filtradas.parquet`, `origenes_validacion.parquet`, `series_kalman.parquet`, escaladores. Modelos: `modelo_A1.pt`, `modelo_A2.pt`, `modelo_B.pt`, `modelo_C.pt`, `modelo_D.pt`, `bi_noa_lstm.pt`, `bi_lstm_baseline.pt` |
| **Salidas** | `reports/metricas_comparativas.csv`, `reports/evaluacion/` (métricas, tiempos, ranking, detalle de celdas críticas, figuras) |

**Modelos (9)**

| Clave | Modelo | Origen |
|---|---|---|
| A1 | BO-VMD + SOMA-Bi-NOA-LSTM, MIMO por electrolizador | Benchmarking 1.º (5.00) |
| A2 | BO-VMD + SOMA-Bi-NOA-LSTM, MIMO conjunto | Benchmarking 1.º (5.00) |
| C | ST-GRU | Benchmarking 2.º (4.00) |
| D | TCN Multi-Output | Benchmarking 3.º (3.90) |
| B | Seq2Seq BiLSTM + atención + Kalman (ensamble neuro-estocástico) | Benchmarking 4.º (3.80) |
| Propio 1 | Bi-NOA-LSTM (canal independiente, ancla, covariables, máscara de saltos) | Nuestro |
| Propio 2 | Bi-LSTM estándar (ídem, con `nn.LSTM`) | Nuestro |
| Persistencia | `ŷ(t0 + h) = y(t0)`: último voltaje real observado | Referencia |
| Tendencia lineal | Recta sobre los 30 días de la serie real sin filtrar | Referencia |

**Protocolo común**
- Orígenes `t0` cada `EVAL_STRIDE` días desde el último día de train (rolling-origin); cada modelo recibe solo información disponible hasta `t0` (VMD causal, Kalman causal o serie real con relleno causal, según su propuesta).
- Predicciones **desnormalizadas** y comparadas contra el **voltaje real sin filtrar**, solo en objetivos con calidad `ok`/`interpolado`, dentro de validación y con ventana de entrada usable. Todos los modelos se evalúan sobre **exactamente los mismos objetivos**.
- Métricas: **MAE, MSE, RMSE, MAPE y R²**, sobre los horizontes 1–180 juntos y a 60 y 180 días; además, separando los objetivos que cruzan un salto de nivel. Tres versiones de R²:
  - **R² global:** todos los objetivos juntos con la media general. Incluye las diferencias de nivel entre celdas, por lo que hasta la persistencia obtiene un valor alto.
  - **R² sobre el cambio (`R2_cambio`):** compara el cambio real `y(t0 + h) − y(t0)` con el predicho `ŷ(t0 + h) − y(t0)`. Mide qué parte de la evolución futura anticipa el modelo; la persistencia queda en ≈ 0 por definición y un valor positivo indica que el modelo anticipa los cambios.
  - **R² por celda (`R2_medio`):** R² de cada celda sobre sus orígenes, promediado. Muy exigente, porque dentro de una celda el voltaje varía poco.

In [ ]:
# Configuración: localiza la raíz del proyecto (local o Colab) y agrega src/ al path
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Poc-TP1")
except ImportError:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "voltcast").is_dir())
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import json
import time
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="All-NaN slice")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from openpyxl.utils import get_column_letter

from voltcast import cleaning as cl
from voltcast import config, io
from voltcast import dataset as ds
from voltcast import kalman as kf
from voltcast import metrics as mt
from voltcast import mimo
from voltcast import model as mdl

config.ensure_dirs()
EVAL_DIR = config.REPORTS_DIR / "evaluacion"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

M_A1, M_A2 = "A1 · MIMO por EL (Bench. 1.º)", "A2 · MIMO conjunto (Bench. 1.º)"
M_C, M_D, M_B = "C · ST-GRU (Bench. 2.º)", "D · TCN (Bench. 3.º)", "B · Seq2Seq-Att + Kalman (Bench. 4.º)"
M_P1, M_P2 = "Propio · Bi-NOA-LSTM", "Propio · Bi-LSTM"
M_PER, M_TEN = "Persistencia", "Tendencia lineal"
ORDEN = [M_A1, M_A2, M_C, M_D, M_B, M_P1, M_P2, M_PER, M_TEN]
BENCH = [M_A1, M_A2, M_C, M_D, M_B]
PROPIOS = [M_P1, M_P2]
BASES = [M_PER, M_TEN]
REDES = BENCH + PROPIOS
COLORES = dict(zip(ORDEN, ["tab:purple", "tab:pink", "tab:olive", "tab:cyan", "tab:green", "tab:blue", "tab:orange",
                           "k", "tab:gray"]))
ESTILO = {m: ("--" if m in BASES else "-") for m in ORDEN}


def savefig(fig, name):
    path = EVAL_DIR / f"{name}.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    print(f"Figura guardada: {path.relative_to(config.BASE_DIR)}")

## 1. Datos de validación

In [ ]:
gold = io.read_layer("gold", "series_filtradas.parquet")
orig = io.read_layer("gold", "origenes_validacion.parquet")
kal = io.read_layer("gold", "series_kalman.parquet")
esc = io.read_json("gold", "escaladores.json")
esc_k = io.read_json("gold", "escaladores_kalman.json")
CELLS = sorted(gold["celda"].astype(str).unique())
GRUPOS = {el: [c for c in CELLS if c[0] == el] for el in ("A", "B")}
L, H, C = config.LOOKBACK_DAYS, config.HORIZON_DAYS, len(CELLS)

V = ds.pivot(gold, "voltaje", CELLS)
FECHAS = V.index
Q = ds.pivot(gold, "calidad", CELLS)
VALIDO = Q.isin(cl.CALIDAD_VALIDA).to_numpy()
OOS = (Q == "fuera_servicio").to_numpy()
KA, VC = (ds.pivot(gold, c, CELLS).to_numpy() for c in ("kA_z", "vcel_z"))
VR = V.to_numpy()
VRV = np.where(VALIDO, VR, np.nan)                  # voltaje real válido
KAL = ds.pivot(kal, "v_kalman", CELLS).reindex(FECHAS).to_numpy()
PEND = ds.pivot(kal, "pendiente_kalman", CELLS).reindex(FECHAS).to_numpy()
VAL_DESDE = gold.loc[gold["split"] == "val", "fecha"].min()

o = orig.assign(celda=orig["celda"].astype(str)).sort_values(["fecha_origen", "celda", "offset"])
ORIGENES = pd.DatetimeIndex(o["fecha_origen"].unique())
O = len(ORIGENES)
W = o["v_filtrada"].to_numpy().reshape(O, C, L)  # ventanas VMD causales [O, C, L]
assert (o["celda"].to_numpy().reshape(O, C, L)[0, :, 0] == np.array(CELLS)).all()

p0 = FECHAS.get_indexer(ORIGENES)
win = p0[:, None] + np.arange(-L + 1, 1)
tgt = p0[:, None] + np.arange(1, H + 1)
dentro = tgt < len(FECHAS)
tgt_c = np.minimum(tgt, len(FECHAS) - 1)
REAL = VR[tgt_c].transpose(0, 2, 1)                 # [O, C, H]
WK = KAL[win].transpose(0, 2, 1)                    # ventanas Kalman
WR = VRV[win].transpose(0, 2, 1)                    # ventanas de la serie real (válida)
entrada_ok = (np.isfinite(W).all(-1) & np.isfinite(WK).all(-1) & ~OOS[win].transpose(0, 2, 1).any(-1)
              & np.isfinite(WR).any(-1))
MASK = VALIDO[tgt_c].transpose(0, 2, 1) & dentro[:, None, :] & entrada_ok[..., None] & np.isfinite(REAL)
n_val = int((FECHAS >= VAL_DESDE).sum())
print(f"{O} orígenes ({ORIGENES[0].date()} → {ORIGENES[-1].date()}) × {C} celdas ({', '.join(f'{k}: {len(v)}' for k, v in GRUPOS.items())}) × {H} horizontes")
print(f"Validación: {n_val} filas diarias ≥ L + H = {L + H} → alcanza para {(tgt[:, -1] < len(FECHAS)).sum()} ventanas completas")
print(f"Objetivos evaluables (comunes a los 9 modelos): {MASK.sum():,}")

In [ ]:
saltos = cl.level_shifts(V, Q.apply(lambda s: pd.Categorical(s, categories=cl.CALIDAD).codes))
NXT = ds.next_jump_index(FECHAS, CELLS, saltos)
CRUZA = NXT[p0][:, :, None] <= tgt[:, None, :]
print(f"Saltos en validación: {(saltos['fecha'] >= VAL_DESDE).sum()} | "
      f"objetivos que cruzan un salto: {(MASK & CRUZA).sum() / MASK.sum():.1%}")

## 2. Inferencia de los 9 modelos (tiempos medidos)

In [ ]:
PRED, T_INF = {}, {}
EL_FLAG = np.array([float(c[0] == "B") for c in CELLS])
IDX = {c: i for i, c in enumerate(CELLS)}


def cronometrar(nombre, fn):
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t = time.time()
    out = fn()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    T_INF[nombre] = (time.time() - t) / O * 1000  # ms por origen (todas las celdas)
    PRED[nombre] = out
    print(f"{nombre}: {T_INF[nombre]:.1f} ms por origen")


# --- Propios: canal independiente sobre la señal VMD causal ---
for nombre, archivo in [(M_P1, "bi_noa_lstm.pt"), (M_P2, "bi_lstm_baseline.pt")]:
    m, ck = mdl.load_checkpoint(config.MODELS_DIR / archivo, DEVICE)
    assert ck["cells"] == CELLS

    def run(m=m):
        out = np.empty((O, C, H))
        for i in range(O):
            out[i], _ = mdl.predict_volts(m, W[i], VALIDO[win[i]].T, KA[win[i]].T, VC[win[i]].T,
                                          esc["sigma_delta"], np.arange(C), EL_FLAG, DEVICE)
        return out
    cronometrar(nombre, run)

# --- B: red Seq2Seq sobre Kalman + ensamble neuro-estocástico con el pronóstico de Kalman ---
m, ck = mdl.load_checkpoint(config.MODELS_DIR / "modelo_B.pt", DEVICE)
ALPHA_B = ck["alpha_fusion"]


def run_b():
    out = np.empty((O, C, H))
    for i in range(O):
        red, _ = mdl.predict_volts(m, WK[i], VALIDO[win[i]].T, KA[win[i]].T, VC[win[i]].T, esc_k["sigma_delta"],
                                   np.arange(C), EL_FLAG, DEVICE)
        kal_f = kf.kalman_forecast(KAL[p0[i]], np.nan_to_num(PEND[p0[i]]), H)
        out[i] = ALPHA_B * red + (1 - ALPHA_B) * kal_f
    return out
cronometrar(M_B, run_b)
print(f"   α del ensamble B = {ALPHA_B:.2f}")

In [ ]:
# --- A1 / A2: MIMO sobre la señal VMD causal + covariables seleccionadas por RFR-SHAP (Min-Max del checkpoint) ---
COV_EL = {el: pd.DataFrame({"kA_z": KA[:, IDX[GRUPOS[el][0]]], "vcel_z": VC[:, IDX[GRUPOS[el][0]]]}) for el in GRUPOS}
CAL = mimo.calendar_features(FECHAS)


def cargar_grupos(archivo):
    ck = torch.load(config.MODELS_DIR / archivo, map_location=DEVICE, weights_only=False)
    red = mdl.GroupModels({g: mdl.build_model(0, 0, ck["cfg_por_grupo"][g]) for g in ck["grupos"]}).to(DEVICE)
    red.load_state_dict(ck["state_dict"])
    return red.eval(), ck


def run_mimo(archivo):
    red, ck = cargar_grupos(archivo)
    out = np.full((O, C, H), np.nan)
    for gi, g in enumerate(ck["grupos"]):
        celdas = ck["celdas_por_grupo"][g]
        j = [IDX[c] for c in celdas]
        csc = mimo.MinMax.from_dict(ck["escaladores"][g]["celdas"])
        ksc = mimo.MinMax.from_dict(ck["escaladores"][g]["covariables"]) if ck["escaladores"][g]["covariables"] else None
        cov = mimo.covariate_matrix(ck["covariables"], COV_EL, CAL, list(g))
        X = np.stack([mimo.scaled_inputs(W[i][j].T, cov[win[i]], csc, ksc) for i in range(O)])  # [O, L, n_in]
        with torch.no_grad():
            y = red(torch.as_tensor(X, device=DEVICE), torch.full((O,), gi, device=DEVICE)).cpu().numpy()
        out[:, j, :] = csc.inverse(y).transpose(0, 2, 1)
    return out


cronometrar(M_A1, lambda: run_mimo("modelo_A1.pt"))
cronometrar(M_A2, lambda: run_mimo("modelo_A2.pt"))

# --- C / D: serie real normalizada con relleno causal ---
VR_FF = mimo.causal_ffill(VRV)


def run_c():
    m, ck = mdl.load_checkpoint(config.MODELS_DIR / "modelo_C.pt", DEVICE)
    out = np.full((O, C, H), np.nan)
    for gi, g in enumerate(ck["grupos"]):
        j = [IDX[c] for c in ck["celdas_por_grupo"][g]]
        csc = mimo.MinMax.from_dict(ck["escaladores"][g]["celdas"])
        ksc = mimo.MinMax.from_dict(ck["escaladores"][g]["kA"])
        v = np.nan_to_num(csc.transform(VR_FF[:, j]))
        k = np.repeat(np.nan_to_num(ksc.transform(KA[:, [j[0]]])), len(j), axis=1)
        Xg = np.stack([v, k], axis=-1).astype(np.float32)
        X = np.stack([Xg[win[i]] for i in range(O)])
        with torch.no_grad():
            y = m(torch.as_tensor(X, device=DEVICE), torch.full((O,), gi, device=DEVICE)).cpu().numpy()
        out[:, j, :] = csc.inverse(y).transpose(0, 2, 1)
    return out


def run_d():
    red, ck = cargar_grupos("modelo_D.pt")
    out = np.full((O, C, H), np.nan)
    for gi, g in enumerate(ck["grupos"]):
        j = [IDX[c] for c in ck["celdas_por_grupo"][g]]
        csc = mimo.MinMax.from_dict(ck["escaladores"][g]["celdas"])
        ksc = mimo.MinMax.from_dict(ck["escaladores"][g]["kA"])
        Xg = mimo.scaled_inputs(VR_FF[:, j], KA[:, [j[0]]], csc, ksc)
        X = np.stack([Xg[win[i]] for i in range(O)])
        with torch.no_grad():
            y = red(torch.as_tensor(X, device=DEVICE), torch.full((O,), gi, device=DEVICE)).cpu().numpy()
        out[:, j, :] = csc.inverse(y).transpose(0, 2, 1)
    return out


cronometrar(M_C, run_c)
cronometrar(M_D, run_d)

# --- Baselines sobre la serie real sin filtrar ---
cronometrar(M_PER, lambda: mt.persistence(mt.last_observed(WR), H))
cronometrar(M_TEN, lambda: mt.linear_trend(WR, H))
PRED = {k: PRED[k] for k in ORDEN}
assert all(np.isfinite(p[MASK]).all() for p in PRED.values()), "Algún modelo tiene predicciones no finitas en objetivos evaluables"

## 3. Tabla comparativa (`metricas_comparativas.csv`)

In [ ]:
SUB = {"todos": MASK, "sin salto": MASK & ~CRUZA, "cruza salto": MASK & CRUZA}
REF = mt.last_observed(WR)  # voltaje real en t0 (o el último disponible): referencia del R² sobre el cambio
tablas = [mt.summary_at(PRED, REAL, mk, h, ref=REF).assign(subconjunto=s) for s, mk in SUB.items()
          for h in [None] + config.EVAL_HORIZONS]
metricas = pd.concat(tablas, ignore_index=True)
metricas["horizonte"] = metricas["horizonte"].astype(str)
METR = ["MAE_V", "MSE_V2", "RMSE_V", "MAPE_%", "R2_global", "R2_cambio", "R2_medio"]
MAYOR_MEJOR = {"R2_global", "R2_cambio", "R2_medio", "R2_mediana"}
for col in METR:
    g = metricas.groupby(["subconjunto", "horizonte"])[col]
    best = g.transform("max") if col in MAYOR_MEJOR else g.transform("min")
    metricas[f"mejor_{col}"] = np.isclose(metricas[col], best)
metricas.to_csv(config.REPORTS_DIR / "metricas_comparativas.csv", index=False)
metricas.to_csv(config.REPORTS_DIR / "metricas.csv", index=False)
print(f"✔ {(config.REPORTS_DIR / 'metricas_comparativas.csv').relative_to(config.BASE_DIR)}")


def vista(sub="todos", h="1–180", cols=METR):
    t = metricas.query("subconjunto == @sub and horizonte == @h").set_index("modelo").reindex(ORDEN)[cols]
    sty = t.style.format({c: "{:.6f}" if c == "MSE_V2" else "{:.4f}" for c in cols})
    for c in cols:
        sty = sty.highlight_max(subset=[c], color="#c6efce") if c in MAYOR_MEJOR else sty.highlight_min(subset=[c], color="#c6efce")
    return sty


vista("todos", "1–180")

In [ ]:
vista("todos", "60")

In [ ]:
vista("todos", "180")

In [ ]:
mejores = (metricas.melt(id_vars=["subconjunto", "horizonte", "modelo"], value_vars=[f"mejor_{c}" for c in METR],
                         var_name="metrica", value_name="es_mejor").query("es_mejor"))
mejores["metrica"] = mejores["metrica"].str.replace("mejor_", "")
(mejores.pivot_table(index=["subconjunto", "horizonte"], columns="metrica", values="modelo",
                     aggfunc=lambda s: " / ".join(s)).reindex(columns=METR))

## 4. Mejora frente a los baselines (MAE y RMSE)

In [ ]:
filas = []
for h in ["1–180"] + [str(x) for x in config.EVAL_HORIZONS]:
    t = metricas.query("subconjunto == 'todos' and horizonte == @h").set_index("modelo")
    for m in REDES:
        fila = {"horizonte": h, "modelo": m}
        for b in BASES:
            for met in ("MAE_V", "RMSE_V"):
                fila[f"{met[:-2]} vs {b} (%)"] = 100 * (t.loc[b, met] - t.loc[m, met]) / t.loc[b, met]
        fila["supera a ambos baselines (MAE y RMSE)"] = all(fila[k] > 0 for k in fila if k.endswith("(%)"))
        filas.append(fila)
mejora = pd.DataFrame(filas)
mejora.to_csv(EVAL_DIR / "mejora_vs_baselines.csv", index=False)
no_supera = mejora[~mejora["supera a ambos baselines (MAE y RMSE)"]]
print("Modelos que NO superan a ambos baselines en MAE y RMSE:")
print(no_supera[["horizonte", "modelo"]].to_string(index=False) if len(no_supera) else "   ninguno")
mejora.set_index(["horizonte", "modelo"]).round(1)

## 5. ¿Nuestro modelo supera a los del Benchmarking? (significancia)

In [ ]:
# Bootstrap por orígenes del ΔRMSE: cada modelo propio frente al mejor modelo del Benchmarking en ese horizonte
filas = []
for h in config.EVAL_HORIZONS:
    hs = str(h)
    t = metricas.query("subconjunto == 'todos' and horizonte == @hs").set_index("modelo")
    rival = t.loc[BENCH, "RMSE_V"].idxmin()
    for m in PROPIOS:
        d, ci = mt.bootstrap_rmse_diff(PRED[m], PRED[rival], REAL, MASK, h)
        filas.append({"horizonte": h, "modelo propio": m, "mejor del Benchmarking": rival, "ΔRMSE (V)": d,
                      "IC95 inf": ci[0], "IC95 sup": ci[1],
                      "conclusión": "propio mejor (significativo)" if ci[1] < 0 else
                                    ("propio peor (significativo)" if ci[0] > 0 else "sin diferencia significativa")})
sig = pd.DataFrame(filas)
sig.to_csv(EVAL_DIR / "significancia_propios_vs_benchmarking.csv", index=False)
sig.round(4)

### 5.1 Diagnóstico: sesgo de nivel a 1 día

Si un modelo comete un error grande ya a 1 día, el problema no es el horizonte sino el **nivel**. Se compara el sesgo medio (predicho − real) y el RMSE a 1 día en validación, y para la Propuesta A1 también en orígenes del último año de **train** (con su señal VMD de train), para distinguir un error de código de un problema de generalización.

In [ ]:
\
diag = pd.DataFrame({n: {"sesgo h1 (V)": float(np.nanmean(np.where(MASK[..., 0], p[..., 0] - REAL[..., 0], np.nan))),
                         "RMSE h1 (V)": float(mt.rmse(p[..., 0], REAL[..., 0], MASK[..., 0]))} for n, p in PRED.items()}).T
print("Validación:")
print(diag.round(4).to_string())

# A1 en orígenes de train (señal VMD de train, sin ver validación)
red, ck = cargar_grupos("modelo_A1.pt")
VF_tr = ds.pivot(gold, "v_filtrada", CELLS).to_numpy()
n_tr = int((FECHAS < VAL_DESDE).sum())
filas = []
for t0 in range(n_tr - 360, n_tr - 1, 30):
    for gi, g in enumerate(ck["grupos"]):
        j = [IDX[c] for c in ck["celdas_por_grupo"][g]]
        csc = mimo.MinMax.from_dict(ck["escaladores"][g]["celdas"])
        ksc = mimo.MinMax.from_dict(ck["escaladores"][g]["covariables"])
        cov = mimo.covariate_matrix(ck["covariables"], COV_EL, CAL, list(g))
        x = mimo.scaled_inputs(VF_tr[t0 - L + 1:t0 + 1][:, j], cov[t0 - L + 1:t0 + 1], csc, ksc)
        with torch.no_grad():
            y = red(torch.as_tensor(x[None], device=DEVICE), torch.full((1,), gi, device=DEVICE)).cpu().numpy()[0]
        e = csc.inverse(y)[0] - VR[t0 + 1, j]
        ok = VALIDO[t0 + 1, j] & np.isfinite(VF_tr[t0 - L + 1:t0 + 1][:, j]).all(0)
        filas.append({"origen (train)": FECHAS[t0].date(), "EL": g, "sesgo h1 (V)": e[ok].mean(),
                      "RMSE h1 (V)": np.sqrt((e[ok] ** 2).mean())})
print("\nA1 en orígenes de train:")
pd.DataFrame(filas).groupby("origen (train)")[["sesgo h1 (V)", "RMSE h1 (V)"]].mean().round(4)

**Lectura.**
- **Nuestras variantes son las mejores en todas las métricas sobre el horizonte completo (1–180 días):** la Bi-LSTM propia tiene el menor MAE (0,0270 V), MSE, RMSE (0,0585 V), MAPE (0,87 %) y el mayor R² (0,75 global). La Bi-NOA-LSTM propia queda segunda en casi todo.
- **A 180 días** nuestras variantes ganan con claridad a todos los modelos del Benchmarking (RMSE 0,069 V frente a 0,098 V del mejor, C · ST-GRU) y la diferencia es **significativa** (IC 95 % de ΔRMSE ≈ −0,036 a −0,021 V).
- **A 60 días** ganan en MAE, MAPE y R² medio, pero en RMSE la **persistencia** queda levemente por delante (0,0574 frente a 0,0582 V), y la ventaja sobre el mejor del Benchmarking (B · Seq2Seq) **no es significativa** (el IC incluye el cero).
- **Los 5 modelos del Benchmarking no superan a la persistencia** en ningún horizonte (MAE y RMSE peores). B y C son los más cercanos; A1, A2 y D quedan por detrás incluso de la tendencia lineal en varias métricas.
- **Por qué fallan A1, A2 y D (diagnóstico 5.1):** fallan ya **a 1 día** (RMSE ≈ 0,10–0,14 V, frente a 0,018 V de la persistencia). A1 y A2 tienen un sesgo medio de +0,05 V; en D el error de nivel va celda por celda en ambos sentidos. En train, en cambio, A1 predice bien el día siguiente (RMSE ≈ 0,011–0,017 V en orígenes del último año). Es un problema de generalización, no un error de código: estos modelos aprenden el **voltaje absoluto** de cada celda (Min-Max) con solo ~1 600 matrices de entrenamiento, y cuando el nivel de las celdas cambia, regresan a patrones típicos de train. Nuestras variantes pronostican **relativo al nivel reciente (ancla)**, lo que evita ese problema.
- **R² sobre el cambio de voltaje (`R2_cambio`):** muestra el límite real de todos los modelos. **Ninguno anticipa de forma útil cómo cambiará cada celda:** el mejor valor es ≈ +0,02–0,04 (Bi-LSTM y Bi-NOA-LSTM propias a 180 días); sobre el horizonte completo nuestra Bi-LSTM queda en −0,03, apenas mejor que la persistencia (−0,09), y a 60 días la persistencia es algo mejor (−0,05 frente a −0,07). Los modelos del Benchmarking quedan muy por debajo (B −0,67; C −1,32; A1/A2 ≈ −3; D −5,6). Conclusión: el **R² global de ≈ 0,75 proviene casi todo de acertar el nivel de cada celda**; los cambios de cada celda están dominados por intervenciones y por la variación diaria (incluido el ruido del valor de referencia en t0), y lo que los modelos capturan es la deriva media.
- **Cruza salto (2,7 % de los objetivos):** todos fallan por 0,19–0,34 V. Aquí los modelos del Benchmarking y la persistencia quedan algo mejor que los nuestros: nuestras variantes proyectan la subida por degradación, y el salto es una caída.

## 6. Tiempos de entrenamiento e inferencia

In [ ]:
def minutos(archivo):
    ck = torch.load(config.MODELS_DIR / archivo, map_location="cpu", weights_only=False)
    t = ck.get("tiempo_min", {})
    return t.get("entrenamiento_min", np.nan), t.get("soma_min", 0.0)


bloque = pd.read_csv(config.REPORTS_DIR / "modelado" / "metricas_bloque_interno.csv", index_col=0)
tiempos = {M_A1: minutos("modelo_A1.pt"), M_A2: minutos("modelo_A2.pt"), M_B: minutos("modelo_B.pt"),
           M_C: minutos("modelo_C.pt"), M_D: minutos("modelo_D.pt"),
           M_P1: (bloque.loc["Bi-NOA-LSTM", "tiempo (s)"] / 60, 0.0),
           M_P2: (bloque.loc["Bi-LSTM estándar", "tiempo (s)"] / 60, 0.0), M_PER: (0.0, 0.0), M_TEN: (0.0, 0.0)}
tt = pd.DataFrame({"entrenamiento (min)": {m: v[0] for m, v in tiempos.items()},
                   "búsqueda SOMA (min)": {m: v[1] for m, v in tiempos.items()},
                   "inferencia (ms por origen, todas las celdas)": T_INF}).reindex(ORDEN)
tt["inferencia (ms por celda)"] = tt["inferencia (ms por origen, todas las celdas)"] / C
tt.to_csv(EVAL_DIR / "tiempos.csv")
tt.round(2)

## 7. Error vs. horizonte (*error drift*)

In [ ]:
curvas = {s: pd.DataFrame({n: mt.rmse(p, REAL, mk, axis=(0, 1)) for n, p in PRED.items()},
                          index=pd.RangeIndex(1, H + 1, name="h")) for s, mk in SUB.items()}
pd.concat(curvas, axis=1).to_csv(EVAL_DIR / "rmse_por_horizonte.csv")
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for ax, (s, cur) in zip(axes, curvas.items()):
    for n in ORDEN:
        ax.plot(cur.index, cur[n], color=COLORES[n], ls=ESTILO[n], lw=1.6 if n in PROPIOS else 1, label=n)
    for h in config.EVAL_HORIZONS:
        ax.axvline(h, color="grey", ls=":", lw=0.8)
    ax.set(xlabel="horizonte (días)", ylabel="RMSE (V)", title=f"Error vs. horizonte — {s}")
axes[0].set_ylim(0, curvas["todos"].drop(columns=M_TEN).max().max() * 1.15)
axes[1].set_ylim(0, curvas["sin salto"].drop(columns=M_TEN).max().max() * 1.15)
axes[0].legend(fontsize=7)
fig.tight_layout(); savefig(fig, "01_error_vs_horizonte")
curvas["todos"].iloc[[0, 6, 29, 59, 89, 119, 179]].round(4)

**Lectura.**
- **Error drift del modelo paso a paso (B):** su RMSE crece de 0,014 V a 1 día a 0,104 V a 180 días, la pendiente más fuerte entre las redes. A 1 día es de los mejores, pero se degrada con el horizonte. Es la **acumulación de error por recursión** que anticipaba el Benchmarking (riesgo «alto» para la Propuesta B).
- **Nuestras variantes** (salida directa) crecen de forma suave hasta 0,069 V a 180 días y quedan por debajo de la persistencia a partir de ~90 días; sin saltos, por debajo en casi todo el rango.
- **A1, A2 y D** son casi planos (≈ 0,10–0,16 V en todo el horizonte): su error está dominado por el sesgo de nivel inicial, no por el horizonte.
- **C (ST-GRU)** empieza mejor que A y D (≈ 0,075 V a 1 día) y sube hasta ≈ 0,098 V.

## 8. Real vs. predicho en celdas críticas (los 9 modelos)

In [ ]:
nivel = pd.Series(np.nanmean(np.where(MASK[..., 0], REAL[..., 0], np.nan), axis=0), index=CELLS)
err180 = pd.Series(mt.rmse(PRED[M_P1][..., 179], REAL[..., 179], MASK[..., 179], axis=0), index=CELLS)
criticas = {"mayor voltaje": nivel.idxmax(), "mayor error a 180 d": err180.idxmax()}
print({k: f"{c} (V medio {nivel[c]:.3f} V)" for k, c in criticas.items()})

fig, axes = plt.subplots(len(criticas), 2, figsize=(17, 4.2 * len(criticas)), sharey="row")
for r, (tit, c) in enumerate(criticas.items()):
    j = IDX[c]
    for col, h in enumerate(config.EVAL_HORIZONS):
        ax = axes[r, col]
        ok = MASK[:, j, h - 1]
        f = ORIGENES[ok] + pd.Timedelta(days=h)
        s = V[c][(V.index >= f.min() - pd.Timedelta(days=30)) & (V.index <= f.max())]
        ax.plot(s.index, s, color="lightgrey", lw=2.5, label="real")
        for n in ORDEN:
            ax.plot(f, PRED[n][ok, j, h - 1], color=COLORES[n], ls=ESTILO[n], lw=1.6 if n in PROPIOS else 0.9,
                    marker=".", ms=3, label=n)
        ax.set(title=f"{c} — {tit} — h = {h} días", ylabel="V")
axes[0, 0].legend(fontsize=7, ncol=2)
fig.tight_layout(); savefig(fig, "02_real_vs_predicho_criticas")

In [ ]:
filas = []
for tit, c in criticas.items():
    j = IDX[c]
    for h in config.EVAL_HORIZONS:
        k = h - 1
        mk, r = MASK[:, j, k], REAL[:, j, k]
        for n, p in PRED.items():
            pk = p[:, j, k]
            filas.append({"celda": f"{c} ({tit})", "horizonte": h, "modelo": n, "MAE (V)": float(mt.mae(pk, r, mk)),
                          "MSE (V²)": float(mt.mse(pk, r, mk)), "RMSE (V)": float(mt.rmse(pk, r, mk)),
                          "MAPE (%)": float(mt.mape(pk, r, mk)),
                          "R²": float(mt.r2_per_cell(pk[:, None], r[:, None], mk[:, None])[0])})
cuadro = pd.DataFrame(filas)
cuadro.to_csv(EVAL_DIR / "metricas_celdas_criticas.csv", index=False)
cuadro.set_index(["celda", "horizonte", "modelo"]).round(4)

In [ ]:
# Detalle por fecha: predicho vs. real, error y error % de los 9 modelos (con la ubicación en el Excel)
det = []
for tit, c in criticas.items():
    j = IDX[c]
    n_c = int(c[3:])
    oi, hi = np.nonzero(MASK[:, j, :])
    d = pd.DataFrame({"celda": c, "electrolizador": c[0], "celda_excel": n_c, "columna_excel": get_column_letter(3 + n_c),
                      "fecha_origen": ORIGENES[oi], "horizonte": hi + 1,
                      "fecha_objetivo": ORIGENES[oi] + pd.to_timedelta(hi + 1, unit="D"),
                      "real_V": REAL[oi, j, hi], "cruza_salto": CRUZA[oi, j, hi]})
    for n, p in PRED.items():
        d[f"pred_V · {n}"] = p[oi, j, hi]
        d[f"error_V · {n}"] = d[f"pred_V · {n}"] - d["real_V"]
        d[f"error_% · {n}"] = 100 * d[f"error_V · {n}"] / d["real_V"]
    det.append(d)
detalle = pd.concat(det, ignore_index=True)
detalle.to_csv(EVAL_DIR / "detalle_celdas_criticas.csv", index=False)
print(f"✔ reports/evaluacion/detalle_celdas_criticas.csv ({len(detalle):,} filas; no se sube al repositorio)")
detalle[detalle["horizonte"].isin(config.EVAL_HORIZONS)].set_index(
    ["celda", "horizonte", "fecha_origen", "fecha_objetivo"]).sort_index().round(4).head(10)

## 9. Error por celda: nuestro mejor modelo frente al mejor del Benchmarking

In [ ]:
tg = metricas.query("subconjunto == 'todos' and horizonte == '1–180'").set_index("modelo")
mejor_propio = tg.loc[PROPIOS, "RMSE_V"].idxmin()
mejor_bench = tg.loc[BENCH, "RMSE_V"].idxmin()
rm = {n: mt.rmse(PRED[n], REAL, MASK, axis=(0, 2)) for n in (mejor_propio, mejor_bench)}
dif = pd.Series(rm[mejor_propio] - rm[mejor_bench], index=CELLS)
fig, ax = plt.subplots(figsize=(15, 2.8))
mat = pd.DataFrame({el: dif[GRUPOS[el]].to_numpy() for el in GRUPOS}).T
sns.heatmap(mat, cmap="RdBu_r", center=0, robust=True, ax=ax, xticklabels=10, cbar_kws={"label": "Δ RMSE (V)"})
ax.set(xlabel="posición de la celda", ylabel="electrolizador",
       title=f"RMSE por celda (h 1–180): {mejor_propio} − {mejor_bench} (azul = gana el propio)")
fig.tight_layout(); savefig(fig, "03_error_por_celda")
print(f"{mejor_propio} gana a {mejor_bench} en {(dif < 0).mean():.0%} de las celdas")

## 10. Cumplimiento de los criterios de `docs/metricas.md`

**Requisito mínimo (RNF06):** MAE < 5 % del voltaje nominal de una celda, MSE < 0,0225 V² (equivale a RMSE < 0,15 V), RMSE < 0,15 V y MAPE < 5 %. El voltaje nominal se toma como la media de las celdas en validación.

**Criterios comparativos** (los que permiten afirmar que un modelo es mejor):
- **C1:** supera a la persistencia y a la tendencia lineal en MAE y RMSE.
- **C2:** RMSE menor que el del mejor modelo del Benchmarking, con significancia estadística (IC 95 % por bootstrap de orígenes). Se evalúa para nuestras variantes.
- **C3:** RMSE < 0,15 V en al menos el 95 % de las celdas.
- **C4:** R² sobre el cambio de voltaje > 0 (anticipa la evolución mejor que el cambio promedio).

Se evalúan sobre todos los objetivos, en el horizonte completo y a 60 y 180 días.

In [ ]:
NOM = float(np.nanmean(np.where(MASK, REAL, np.nan)))
UMBRAL = {"MAE_V": 0.05 * NOM, "MSE_V2": 0.15 ** 2, "RMSE_V": 0.15, "MAPE_%": 5.0}
print(f"Voltaje nominal (media de las celdas en validación) = {NOM:.3f} V → MAE < {UMBRAL['MAE_V']:.3f} V")

# RMSE y MAE por celda (horizontes 60 y 180) para C3
por_celda = []
for n, p in PRED.items():
    for h in config.EVAL_HORIZONS:
        k = h - 1
        por_celda.append(pd.DataFrame({"celda": CELLS, "modelo": n, "horizonte": h,
                                       "RMSE_V": mt.rmse(p[..., k], REAL[..., k], MASK[..., k], axis=0),
                                       "MAE_V": mt.mae(p[..., k], REAL[..., k], MASK[..., k], axis=0)}))
por_celda = pd.concat(por_celda, ignore_index=True)
por_celda.to_csv(EVAL_DIR / "metricas_por_celda.csv", index=False)
rmse_celda_1_180 = {n: mt.rmse(p, REAL, MASK, axis=(0, 2)) for n, p in PRED.items()}

filas = []
for h in ["1–180"] + [str(x) for x in config.EVAL_HORIZONS]:
    t = metricas.query("subconjunto == 'todos' and horizonte == @h").set_index("modelo")
    mej = mejora.query("horizonte == @h").set_index("modelo")
    for n in ORDEN:
        fila = {"horizonte": h, "modelo": n}
        for met, u in UMBRAL.items():
            fila[f"req · {met} < {u:.4g}"] = bool(t.loc[n, met] < u)
        fila["requisito mínimo (todos)"] = all(v for k, v in fila.items() if k.startswith("req ·"))
        fila["C1 supera baselines"] = bool(mej.loc[n, "supera a ambos baselines (MAE y RMSE)"]) if n in REDES else None
        if n in PROPIOS and h != "1–180":
            fila["C2 supera al mejor Bench. (sig.)"] = bool(
                sig.query("horizonte == @h_int and `modelo propio` == @n", local_dict={"h_int": int(h), "n": n})["IC95 sup"].item() < 0)
        else:
            fila["C2 supera al mejor Bench. (sig.)"] = None
        r = (rmse_celda_1_180[n] if h == "1–180" else
             por_celda.query("modelo == @n and horizonte == @hi", local_dict={"n": n, "hi": int(h)})["RMSE_V"].to_numpy())
        fila["C3 % celdas RMSE < 0,15 V"] = float(np.nanmean(r < 0.15) * 100)
        fila["C3 cumple (≥ 95 %)"] = fila["C3 % celdas RMSE < 0,15 V"] >= 95
        fila["C4 R² cambio > 0"] = bool(t.loc[n, "R2_cambio"] > 0)
        filas.append(fila)
cumpl = pd.DataFrame(filas)
cumpl.to_csv(EVAL_DIR / "cumplimiento_metricas.csv", index=False)
print("✔ reports/evaluacion/cumplimiento_metricas.csv y metricas_por_celda.csv")


def marca(v):
    return "—" if v is None or (isinstance(v, float) and np.isnan(v)) else ("✔" if v is True else ("✘" if v is False else f"{v:.0f} %"))


resumen = cumpl.set_index(["horizonte", "modelo"])[["requisito mínimo (todos)", "C1 supera baselines",
                                                     "C2 supera al mejor Bench. (sig.)", "C3 % celdas RMSE < 0,15 V",
                                                     "C4 R² cambio > 0"]]
resumen.map(marca)

**Lectura.**
- **Requisito mínimo (RNF06):** lo cumplen **7 de los 9 modelos** en todos los horizontes, incluida la persistencia. Solo fallan D · TCN y la tendencia lineal, en RMSE a 180 días. Confirma que estos umbrales verifican un error aceptable, pero no distinguen a un modelo de otro.
- **Nuestra Bi-LSTM** frente a los criterios comparativos:

| Horizonte | C1 supera baselines | C2 supera al mejor Bench. (sig.) | C3 ≥ 95 % de celdas | C4 R² cambio > 0 |
|---|---|---|---|---|
| 1–180 días | ✔ | — | ✔ (98,3 %) | ✘ (−0,03) |
| 60 días | ✘ (la persistencia gana en RMSE) | ✘ (sin significancia frente a B) | ✔ (96,7 %) | ✘ |
| 180 días | ✔ | ✔ | ✘ (94,0 %, justo por debajo) | ✔ (+0,02) |

- **Ningún modelo del Benchmarking cumple C1** en ningún horizonte, y ninguno cumple C4. En C3, B es el único cercano a nuestras variantes (96,3 % sobre el horizonte completo); A1, A2 y D quedan en 70–82 %.
- **Conclusión:** a **180 días** nuestro modelo cumple los criterios que demuestran superioridad (C1, C2 y C4) y queda a un punto de C3; a **60 días** no se puede afirmar que supere a la persistencia ni al mejor modelo del Benchmarking.

## 11. Ranking empírico frente al Benchmarking

In [ ]:
rank = pd.DataFrame({"puesto Benchmarking": pd.Series({M_A1: "1.º (5.00)", M_A2: "1.º (5.00)", M_C: "2.º (4.00)",
                                                       M_D: "3.º (3.90)", M_B: "4.º (3.80)", M_P1: "—", M_P2: "—",
                                                       M_PER: "—", M_TEN: "—"})})
for h in ["1–180", "60", "180"]:
    t = metricas.query("subconjunto == 'todos' and horizonte == @h").set_index("modelo")
    rank[f"RMSE {h} (V)"] = t["RMSE_V"]
    rank[f"puesto RMSE {h}"] = t["RMSE_V"].rank().astype(int)
rank["puesto MAE 1–180"] = tg["MAE_V"].rank().astype(int)
rank = rank.reindex(ORDEN).sort_values("puesto RMSE 1–180")
rank.to_csv(EVAL_DIR / "ranking.csv")
orden_bench = [M_A1, M_C, M_D, M_B]
emp = rank.loc[orden_bench, "puesto RMSE 1–180"].sort_values().index.tolist()
print("Orden del Benchmarking (A, C, D, B):", " > ".join(m.split(" · ")[0] for m in orden_bench))
print("Orden empírico (RMSE 1–180):        ", " > ".join(m.split(" · ")[0] for m in emp))
print("¿Coincide?", emp == orden_bench)
rank.round(4)

## Conclusión: ranking empírico frente al Benchmarking

**Ranking empírico (RMSE 1–180 días):** Propio · Bi-LSTM (0,0585 V) > Propio · Bi-NOA-LSTM (0,0601) > Persistencia (0,0602) > B · Seq2Seq (0,0744) > C · ST-GRU (0,0879) > A2 (0,1143) > Tendencia lineal (0,1179) > A1 (0,1186) > D · TCN (0,1487).

**El ranking del Benchmarking no se confirma con los datos reales.** La literatura ordenaba A > C > D > B; empíricamente queda **B > C > A > D**: la Propuesta A, ganadora del Benchmarking, es de las peores, y la B, la última, es la mejor de las cuatro. El Benchmarking valoró las arquitecturas por la literatura (otros datos, horarios, con más muestras); con 1 945 días de train y un nivel de celda que cambia con el tiempo, los modelos MIMO de voltaje absoluto generalizan mal.

**Nuestro modelo frente al Benchmarking:**
- **Gana en todas las métricas** (MAE, MSE, RMSE, MAPE, R² global y sobre el cambio) sobre el horizonte completo y a 180 días, con significancia estadística a 180 días.
- A 60 días gana en MAE, MAPE y R², pero sin significancia en RMSE, y la persistencia queda levemente mejor en RMSE.
- Es **el único grupo de redes que supera a ambos baselines** (1–180 y 180 días).
- Es de los más rápidos: la Bi-LSTM propia entrena en ~2 min e infiere en ~2 ms por origen (300 celdas); la Propuesta A necesitó ~3 h con SOMA.

**Qué explica la ventaja (contribuciones propias):** (1) **pronóstico relativo al ancla** (nivel reciente) en lugar de voltaje absoluto, (2) **una celda por muestra con pesos compartidos**, que multiplica por 300 los ejemplos de entrenamiento, y (3) **máscara de saltos de nivel**, que evita sobreajustar a las intervenciones.

**Alcance real del resultado:** el R² sobre el cambio de voltaje (≈ 0) indica que, con los datos disponibles, ningún modelo anticipa la evolución específica de cada celda mejor que su deriva media. La ventaja de nuestro modelo es **seguir mejor el nivel y la deriva** que los demás (menor error en voltios), no predecir los cambios individuales. Para mejorar eso hacen falta variables que hoy no están en el dataset, sobre todo el registro de mantenimientos.

**Elección del modelo:** la **Bi-LSTM propia** es la candidata: mejor o igual que la Bi-NOA-LSTM en casi todas las métricas, ~2 veces más rápida de entrenar y ~13 veces más rápida en inferencia. La celda NOA no aporta ventaja en estos datos.

**Limitaciones:**
- Una sola semilla por modelo: las diferencias pequeñas (p. ej. Bi-LSTM frente a Bi-NOA-LSTM) no son concluyentes.
- A 60 días, ningún modelo supera con claridad a la persistencia.
- Las intervenciones de mantenimiento (2,7 % de los objetivos) siguen sin ser predecibles; para eso hace falta el registro de mantenimientos de la planta.